# Training-free guidance experiments

This notebook reproduces the guidance experiments: an unguided baseline and
four guided motifs (3D-printed steel,
cast metal slab, copper weld, and heterogeneous lamella titanium). It intentionally
contains only these four guidance objectives.

All generated rows use the same four initial Gaussian states (seeds 117, 237,
377, and 527), 300 generators, 60 Euler steps, a 16-nearest-neighbour graph,
and a 10% unguided warm-up followed by a cosine guidance ramp. The rightmost
images are qualitative experimental references only; they are never loaded by
the model or used by any objective.

Run `python prepare_artifacts.py` once from the repository root before executing
this notebook.


In [ ]:
from pathlib import Path
import math

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import PyAPD
import torch

from helpers import (
    build_knn_graph,
    denormalise_sample,
    expm_sym,
    reconstruct_symmetric_2x2,
    sample_base_like,
)
from model import FlowEGNN, load_model_checkpoint
from sample import compute_apd


## Configuration

These are the architecture and sampling settings used for the guidance
figure. The supplied checkpoint was trained only on synthetic APDs.


In [ ]:
ROOT = Path.cwd()
CHECKPOINT_PATH = (
    ROOT / "model_for_testing" / "model_256hd_3ly_16knn_100_500_N100.pt"
)
STATS_PATH = ROOT / "data_for_testing" / "12k_N100_500_stats.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_SEEDS = [117, 237, 377, 527]
NUM_NODES = 300
K_KNN = 16
HIDDEN_DIM = 256
N_LAYERS = 3

N_STEPS = 60
SAVE_EVERY = 10
KNN_UPDATE_EVERY = 1
SCHEDULE_START = 0.10

PLOT_PIXEL_SIZE = 250
SAVE_MOTIF_FIGURE = True
MOTIF_FIGURE_PATH = (
    ROOT / "examples_for_guidance" / "guidance_examples.png"
)

for required_path in (CHECKPOINT_PATH, STATS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print(f"device: {DEVICE}")
print(f"checkpoint: {CHECKPOINT_PATH.relative_to(ROOT)}")
print(f"statistics: {STATS_PATH.relative_to(ROOT)}")
# The exact run count is printed after the unified motif list is defined.


In [ ]:
model_state, checkpoint_payload = load_model_checkpoint(CHECKPOINT_PATH)

if checkpoint_payload is not None:
    model_config = checkpoint_payload.get("model_config", {})
    HIDDEN_DIM = model_config.get("hidden_dim", HIDDEN_DIM)
    N_LAYERS = model_config.get("n_layers", N_LAYERS)
    K_KNN = model_config.get("k_knn", K_KNN)
    stats = checkpoint_payload.get("normalization_stats")
else:
    stats = None

if stats is None:
    stats = torch.load(STATS_PATH, map_location="cpu", weights_only=False)

stats = {
    key: value.to(DEVICE) if torch.is_tensor(value) else value
    for key, value in stats.items()
}

model = FlowEGNN(hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS)
model.load_state_dict(model_state)
model.to(DEVICE).eval()

# Parameter gradients are unnecessary. Input gradients still pass through the model.
model.requires_grad_(False)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(
    f"loaded {parameter_count:,} parameters "
    f"(hidden_dim={HIDDEN_DIM}, layers={N_LAYERS}, k={K_KNN})"
)


## Guidance method

For linear flow matching, the model predicts a velocity
`v_theta(z_t, t)`. A first-order estimate of its terminal state is

    z_hat_1 = z_t + (1 - t) v_theta(z_t, t).

At each Euler step, every active objective is evaluated on `z_hat_1` and
differentiated with respect to generator positions `x`, anisotropies `ani`, and
power weights `w`. Each objective/channel gradient is RMS-normalized before its
independently configured strength is applied. The state is detached after every
update, so this is local terminal-state guidance rather than backpropagation
through the full trajectory.

The k-nearest-neighbour graph is recomputed at every step. Neighbour selection
is discrete: gradients pass through the model on the current edge set, not
through the selection of that edge set.


In [ ]:
STATE_KEYS = ("x", "ani", "w")


def clone_state(state):
    """Clone tensor values so multiple runs can start from identical noise."""
    return {key: value.detach().clone() for key, value in state.items()}


def make_base_state(num_nodes, k_knn, device):
    """Use the repository's Gaussian training base and device-aware kNN."""
    template = {
        "x": torch.empty(num_nodes, 2, device=device),
        "ani": torch.empty(num_nodes, 2, device=device),
        "w": torch.empty(num_nodes, 1, device=device),
        "batch": torch.zeros(num_nodes, dtype=torch.long, device=device),
        "ptr": torch.tensor([0, num_nodes], dtype=torch.long, device=device),
    }
    state = sample_base_like(template, device=device)
    state["edge_index"] = build_knn_graph(
        state["x"], k=k_knn, batch=state["batch"]
    )
    return state


def terminal_estimate(state, velocity, time_value):
    """First-order prediction of z(1) from the current flow state."""
    remaining_time = 1.0 - time_value
    return {
        key: state[key] + remaining_time * velocity[key]
        for key in STATE_KEYS
    }


def rms_normalize(gradient, epsilon=1e-8):
    """Normalize one channel without changing its gradient direction."""
    rms = gradient.square().mean().sqrt().clamp_min(epsilon)
    return gradient / rms


def cosine_ramp(time_value, start=0.10):
    """Smoothly increase guidance from zero after an initial warm-up."""
    if time_value <= start:
        return 0.0
    progress = min((time_value - start) / (1.0 - start), 1.0)
    return 0.5 - 0.5 * math.cos(math.pi * progress)


def to_physical_state(state, stats):
    """Convert normalized channels to positions, SPD matrices, and weights."""
    physical = denormalise_sample(state, stats)
    log_A = reconstruct_symmetric_2x2(physical["ani"])
    A = expm_sym(log_A)
    return {
        "x": physical["x"].detach().cpu(),
        "A": A.detach().cpu(),
        "w": physical["w"].detach().cpu(),
    }


## Guided Euler sampler

The sampler subtracts the scheduled, normalized loss gradients from the learned
velocity and then takes an Euler step. Empty `guidance_terms` recovers the
unguided sampler.


In [ ]:
def strength_is_nonzero(strength):
    value = torch.as_tensor(strength)
    return bool((value != 0).any())


def scale_guidance_gradient(gradient, strength):
    """Apply scalar or per-component strength with normal broadcasting."""
    scale = torch.as_tensor(
        strength,
        device=gradient.device,
        dtype=gradient.dtype,
    )
    return gradient * scale


def sample_with_guidance_terms(
    model,
    initial_state,
    *,
    guidance_terms,
    n_steps=100,
    save_every=10,
    k_knn=16,
    knn_update_every=1,
    schedule_start=0.10,
):
    """
    Apply independently scaled gradients from zero or more objectives.

    Each guidance term contains:
        name: unique diagnostic name
        objective: callable(state, time_value) -> scalar loss
        strengths: separate x, ani, and w strengths

    A strength may be scalar or per-component. For x, (0.0, value)
    applies only vertical guidance.
    """
    if n_steps <= 0 or save_every <= 0 or knn_update_every <= 0:
        raise ValueError("step counts and update intervals must be positive")
    names = [term["name"] for term in guidance_terms]
    if len(names) != len(set(names)):
        raise ValueError("guidance term names must be unique")

    for term in guidance_terms:
        missing = set(STATE_KEYS) - set(term["strengths"])
        if missing:
            raise ValueError(
                f"{term['name']} is missing strengths for {sorted(missing)}"
            )

    state = clone_state(initial_state)
    dt = 1.0 / n_steps
    trajectory = []
    history = {
        "schedule": [],
        "losses": {name: [] for name in names},
    }

    active_terms = [
        term
        for term in guidance_terms
        if any(
            strength_is_nonzero(term["strengths"][key])
            for key in STATE_KEYS
        )
    ]

    for step_index in range(n_steps):
        time_value = step_index / n_steps
        time_graph = torch.tensor(
            [[time_value]],
            device=state["x"].device,
            dtype=state["x"].dtype,
        )
        schedule_value = cosine_ramp(time_value, start=schedule_start)

        total_correction = {
            key: torch.zeros_like(state[key]) for key in STATE_KEYS
        }

        if active_terms and schedule_value > 0.0:
            differentiable_state = {
                key: state[key].detach().requires_grad_(True)
                for key in STATE_KEYS
            }
            differentiable_state.update(
                batch=state["batch"],
                ptr=state["ptr"],
                edge_index=state["edge_index"],
            )

            velocity_with_graph = model(
                differentiable_state,
                time_graph,
                state["edge_index"],
            )
            endpoint = terminal_estimate(
                differentiable_state,
                velocity_with_graph,
                time_value,
            )
            losses = {
                term["name"]: term["objective"](endpoint, time_value)
                for term in guidance_terms
            }

            for term_index, term in enumerate(active_terms):
                raw_gradients = torch.autograd.grad(
                    losses[term["name"]],
                    tuple(differentiable_state[key] for key in STATE_KEYS),
                    retain_graph=term_index < len(active_terms) - 1,
                )

                for key, raw_gradient in zip(STATE_KEYS, raw_gradients):
                    normalized_gradient = rms_normalize(
                        raw_gradient.detach()
                    )
                    total_correction[key] = (
                        total_correction[key]
                        + scale_guidance_gradient(
                            normalized_gradient,
                            term["strengths"][key],
                        )
                    )

            velocity = {
                key: velocity_with_graph[key].detach() for key in STATE_KEYS
            }
        else:
            with torch.no_grad():
                velocity = model(state, time_graph, state["edge_index"])
                endpoint = terminal_estimate(state, velocity, time_value)
                losses = {
                    term["name"]: term["objective"](endpoint, time_value)
                    for term in guidance_terms
                }

        guided_velocity = {
            key: velocity[key] - schedule_value * total_correction[key]
            for key in STATE_KEYS
        }
        next_values = {
            key: state[key] + dt * guided_velocity[key]
            for key in STATE_KEYS
        }
        completed_step = step_index + 1

        if completed_step < n_steps and completed_step % knn_update_every == 0:
            next_edges = build_knn_graph(
                next_values["x"], k=k_knn, batch=state["batch"]
            )
        else:
            next_edges = state["edge_index"]

        state = {
            **{key: value.detach() for key, value in next_values.items()},
            "batch": state["batch"],
            "ptr": state["ptr"],
            "edge_index": next_edges,
        }

        history["schedule"].append(schedule_value)
        for name, loss in losses.items():
            history["losses"][name].append(float(loss.detach().cpu()))

        if completed_step % save_every == 0 or completed_step == n_steps:
            trajectory.append(
                {
                    **clone_state(state),
                    "time": completed_step / n_steps,
                }
            )

    return state, trajectory, history


## Objective helpers

The anisotropy target is represented as a spin-2 vector, so rotating a visible
long axis by 90 degrees leaves the ellipse unchanged while negating the
two-component anisotropy representation.


In [ ]:
def physical_positions(normalized_state, stats):
    return denormalise_sample(normalized_state, stats)["x"]


def normalized_long_axis_target(
    angle_degrees,
    axis_ratio,
    stats,
    reference,
):
    """Encode visible ellipse long axes as normalized spin-2 vectors."""
    angles = torch.as_tensor(
        angle_degrees,
        device=reference.device,
        dtype=reference.dtype,
    )
    ratios = torch.as_tensor(
        axis_ratio,
        device=reference.device,
        dtype=reference.dtype,
    )
    angles, ratios = torch.broadcast_tensors(angles, ratios)
    theta = torch.deg2rad(angles)
    direction = -torch.stack(
        [torch.cos(2.0 * theta), torch.sin(2.0 * theta)],
        dim=-1,
    )
    magnitude = torch.log(ratios.clamp_min(1.0)).unsqueeze(-1)
    return magnitude * direction / stats["ani_scale"]


def weighted_mean(values, weights, epsilon=1e-8):
    return (values * weights).sum() / weights.sum().clamp_min(epsilon)


def centered_mse(predicted, target):
    """Compare relative power weights without constraining their gauge."""
    predicted = predicted.reshape(-1)
    target = target.reshape(-1)
    predicted = predicted - predicted.mean()
    target = target - target.mean()
    return (predicted - target).square().mean()


def visible_long_axis_degrees(normalized_state, stats):
    """Return visible long-axis angles in [-90, 90) degrees."""
    physical_ani = normalized_state["ani"] * stats["ani_scale"]
    doubled_long_axis = -physical_ani
    angles = 0.5 * torch.atan2(
        doubled_long_axis[:, 1],
        doubled_long_axis[:, 0],
    )
    degrees = torch.rad2deg(angles)
    return torch.remainder(degrees + 90.0, 180.0) - 90.0


def guidance_term(name, objective, *, x=0.0, ani=0.0, w=0.0):
    return {
        "name": name,
        "objective": objective,
        "strengths": {"x": x, "ani": ani, "w": w},
    }


### 1. Alternating build-layer orientation

Six horizontal layers alternate between long-axis targets of 55 and 125 degrees
(a 90-degree build axis with a plus/minus 35-degree tilt). Confidence fades at
the smooth layer transitions.


In [ ]:
class LayeredBuildLoss:
    def __init__(
        self,
        stats,
        *,
        n_layers=6,
        build_axis_degrees=90.0,
        tilt_degrees=35.0,
        axis_ratio=5.5,
        transition_sharpness=12.0,
    ):
        if n_layers <= 0:
            raise ValueError("n_layers must be positive")
        if axis_ratio < 1.0:
            raise ValueError("axis_ratio must be at least 1")
        if transition_sharpness <= 0:
            raise ValueError("transition_sharpness must be positive")
        self.stats = stats
        self.n_layers = n_layers
        self.build_axis_degrees = build_axis_degrees
        self.tilt_degrees = tilt_degrees
        self.axis_ratio = axis_ratio
        self.transition_sharpness = transition_sharpness

    def target_and_confidence(self, normalized_state):
        y = physical_positions(normalized_state, self.stats)[:, 1]
        raw_signal = torch.sin(math.pi * self.n_layers * y)
        layer_signal = torch.tanh(
            self.transition_sharpness * raw_signal
        )
        confidence = layer_signal.abs()
        target_angle = (
            self.build_axis_degrees
            + self.tilt_degrees * layer_signal
        )
        target = normalized_long_axis_target(
            target_angle,
            self.axis_ratio,
            self.stats,
            normalized_state["ani"],
        )
        return target, confidence

    def __call__(self, normalized_state, time_value=None):
        del time_value
        target, confidence = self.target_and_confidence(normalized_state)
        per_node = (
            normalized_state["ani"] - target
        ).square().mean(dim=-1)
        return weighted_mean(per_node, confidence.detach())


### 2. Cast-slab chill, columnar, and central zones

Smooth spatial memberships create a fine chill zone, increasingly elongated
columnar grains, and a coarse equiaxed central zone. Separate losses control
shape, relative power weight, and the one-dimensional generator density.


In [ ]:
class CastSlabGuidance:
    def __init__(
        self,
        stats,
        *,
        chill_end=0.11,
        core_start=0.58,
        transition_sharpness=45.0,
        chill_fraction=0.34,
        columnar_fraction=0.48,
    ):
        if not 0.0 < chill_end < core_start < 1.0:
            raise ValueError("zone boundaries must satisfy 0 < chill < core < 1")
        if chill_fraction <= 0 or columnar_fraction <= 0:
            raise ValueError("zone fractions must be positive")
        if chill_fraction + columnar_fraction >= 1.0:
            raise ValueError("zone fractions must leave a nonzero core")
        self.stats = stats
        self.chill_end = chill_end
        self.core_start = core_start
        self.transition_sharpness = transition_sharpness
        self.chill_fraction = chill_fraction
        self.columnar_fraction = columnar_fraction

    def x_coordinate(self, normalized_state):
        return physical_positions(normalized_state, self.stats)[:, 0]

    def memberships(self, normalized_state):
        x = self.x_coordinate(normalized_state)
        entered_columnar = torch.sigmoid(
            self.transition_sharpness * (x - self.chill_end)
        )
        entered_core = torch.sigmoid(
            self.transition_sharpness * (x - self.core_start)
        )
        raw = torch.stack(
            [
                1.0 - entered_columnar,
                entered_columnar * (1.0 - entered_core),
                entered_core,
            ],
            dim=-1,
        )
        return raw / raw.sum(dim=-1, keepdim=True).clamp_min(1e-8)

    def columnar_progress(self, normalized_state):
        x = self.x_coordinate(normalized_state)
        width = self.core_start - self.chill_end
        return ((x - self.chill_end) / width).clamp(0.0, 1.0)

    def shape_loss(self, normalized_state, time_value=None):
        del time_value
        membership = self.memberships(normalized_state)
        progress = self.columnar_progress(normalized_state)
        columnar_ratio = 2.2 + 7.0 * progress
        columnar_target = normalized_long_axis_target(
            torch.zeros_like(progress),
            columnar_ratio,
            self.stats,
            normalized_state["ani"],
        )
        target = membership[:, 1:2] * columnar_target
        return (normalized_state["ani"] - target).square().mean()

    def weight_loss(self, normalized_state, time_value=None):
        del time_value
        membership = self.memberships(normalized_state)
        progress = self.columnar_progress(normalized_state)
        target = (
            -1.2 * membership[:, 0]
            + (-0.2 + progress) * membership[:, 1]
            + 1.4 * membership[:, 2]
        )
        return centered_mse(normalized_state["w"], target)

    def density_loss(self, normalized_state, time_value=None):
        del time_value
        x = torch.sort(self.x_coordinate(normalized_state)).values
        n = x.numel()
        q = (
            torch.arange(n, device=x.device, dtype=x.dtype) + 0.5
        ) / n
        chill_fraction = self.chill_fraction
        columnar_end = chill_fraction + self.columnar_fraction

        chill_target = self.chill_end * q / chill_fraction
        columnar_target = self.chill_end + (
            (self.core_start - self.chill_end)
            * (q - chill_fraction)
            / self.columnar_fraction
        )
        core_target = self.core_start + (
            (1.0 - self.core_start)
            * (q - columnar_end)
            / (1.0 - columnar_end)
        )
        target = torch.where(
            q < chill_fraction,
            chill_target,
            torch.where(q < columnar_end, columnar_target, core_target),
        )
        return (x - target).square().mean()


### 3. Copper-weld growth field

The target long axis follows a curved, inward-facing field, producing elongated
grains that rise from both sides toward the weld centreline.


In [ ]:
class WeldFusionLoss:
    def __init__(
        self,
        stats,
        *,
        axis_ratio=5.5,
        horizontal_pull=1.9,
        vertical_rise=0.62,
    ):
        if axis_ratio < 1.0:
            raise ValueError("axis_ratio must be at least 1")
        self.stats = stats
        self.axis_ratio = axis_ratio
        self.horizontal_pull = horizontal_pull
        self.vertical_rise = vertical_rise

    def target(self, normalized_state):
        x = physical_positions(normalized_state, self.stats)[:, 0]
        horizontal = (0.5 - x) * self.horizontal_pull
        vertical = torch.full_like(horizontal, self.vertical_rise)
        angle = torch.rad2deg(torch.atan2(vertical, horizontal))
        return normalized_long_axis_target(
            angle,
            self.axis_ratio,
            self.stats,
            normalized_state["ani"],
        )

    def __call__(self, normalized_state, time_value=None):
        del time_value
        target = self.target(normalized_state)
        return (normalized_state["ani"] - target).square().mean()


### 4. Heterogeneous lamellae

The highest relative-weight generators form the coarse population. A smooth
wavy-band field assigns this population to four lamellae while the remaining
generators form the fine matrix. Shape and relative-weight losses distinguish
the two populations.


In [ ]:
def role_partition_by_weight(normalized_state, coarse_fraction):
    """Return coarse indices and a complementary fine mask."""
    weights = normalized_state["w"].reshape(-1)
    centered = weights - weights.mean()
    n_nodes = centered.numel()
    n_coarse = int(round(coarse_fraction * n_nodes))
    n_coarse = max(2, min(n_nodes - 1, n_coarse))
    coarse_indices = torch.topk(
        centered,
        k=n_coarse,
        largest=True,
    ).indices
    fine_mask = torch.ones(
        n_nodes,
        dtype=torch.bool,
        device=centered.device,
    )
    fine_mask[coarse_indices] = False
    return coarse_indices, fine_mask


def two_level_relative_weight_loss(
    normalized_state,
    coarse_fraction,
    coarse_target,
):
    """Constrain a two-level relative-weight distribution without fixing labels."""
    weights = normalized_state["w"].reshape(-1)
    weights = torch.sort(weights - weights.mean()).values
    n_nodes = weights.numel()
    n_coarse = int(round(coarse_fraction * n_nodes))
    n_coarse = max(2, min(n_nodes - 1, n_coarse))
    n_fine = n_nodes - n_coarse
    fine_target = -coarse_target * n_coarse / n_fine
    target = torch.full_like(weights, fine_target)
    target[-n_coarse:] = coarse_target
    return (weights - target).square().mean()


def physical_anisotropy_magnitude(normalized_state, stats):
    physical_ani = normalized_state["ani"] * stats["ani_scale"]
    return physical_ani.norm(dim=-1)


class HeteroLamellarGuidance:
    """Coarse-grain lamellae alternating with a fine-grain matrix."""

    def __init__(
        self,
        stats,
        *,
        coarse_fraction=0.22,
        coarse_weight_target=1.55,
        n_bands=4,
        band_sharpness=18.0,
        wave_amplitude=0.035,
        coarse_axis_ratio=1.6,
    ):
        if not 0.0 < coarse_fraction < 0.5:
            raise ValueError("coarse_fraction must be between 0 and 0.5")
        if n_bands <= 0 or band_sharpness <= 0:
            raise ValueError("band count and sharpness must be positive")
        self.stats = stats
        self.coarse_fraction = coarse_fraction
        self.coarse_weight_target = coarse_weight_target
        self.n_bands = n_bands
        self.band_sharpness = band_sharpness
        self.wave_amplitude = wave_amplitude
        self.coarse_axis_ratio = coarse_axis_ratio

    def roles(self, normalized_state):
        return role_partition_by_weight(
            normalized_state,
            self.coarse_fraction,
        )

    def stripe_probability_and_angle(self, normalized_state):
        x = physical_positions(normalized_state, self.stats)
        wave = self.wave_amplitude * torch.sin(
            2.0 * math.pi * x[:, 0]
        )
        phase = 2.0 * math.pi * self.n_bands * (x[:, 1] - wave)
        threshold = math.cos(math.pi * self.coarse_fraction)
        probability = torch.sigmoid(
            self.band_sharpness * (torch.cos(phase) - threshold)
        )

        slope = (
            self.wave_amplitude
            * 2.0
            * math.pi
            * torch.cos(2.0 * math.pi * x[:, 0])
        )
        tangent_angle = torch.rad2deg(
            torch.atan2(slope, torch.ones_like(slope))
        )
        return probability, tangent_angle

    def band_topology_loss(self, normalized_state, time_value=None):
        del time_value
        probability, _ = self.stripe_probability_and_angle(
            normalized_state
        )
        coarse_indices, fine_mask = self.roles(normalized_state)
        probability = probability.clamp(1e-6, 1.0 - 1e-6)
        coarse_loss = -torch.log(probability[coarse_indices]).mean()
        fine_loss = -torch.log(1.0 - probability[fine_mask]).mean()
        return coarse_loss + fine_loss

    def weight_loss(self, normalized_state, time_value=None):
        del time_value
        return two_level_relative_weight_loss(
            normalized_state,
            self.coarse_fraction,
            self.coarse_weight_target,
        )

    def shape_loss(self, normalized_state, time_value=None):
        del time_value
        _, tangent_angle = self.stripe_probability_and_angle(
            normalized_state
        )
        target = normalized_long_axis_target(
            tangent_angle,
            self.coarse_axis_ratio,
            self.stats,
            normalized_state["ani"],
        )
        coarse_indices, fine_mask = self.roles(normalized_state)
        coarse_loss = (
            normalized_state["ani"][coarse_indices]
            - target[coarse_indices]
        ).square().mean()
        fine_magnitude = physical_anisotropy_magnitude(
            normalized_state,
            self.stats,
        )[fine_mask]
        return coarse_loss + fine_magnitude.square().mean()




## Guidance strengths

The following strengths are the values used for the four motifs. A tuple
applies a separate strength to the horizontal and vertical components of `x`.


In [ ]:
LPBF_X_STRENGTH = (0.0, 0.03)
LPBF_ANI_STRENGTH = 18.0

CAST_SHAPE_X_STRENGTH = (0.02, 0.0)
CAST_ANI_STRENGTH = 16.0
CAST_WEIGHT_X_STRENGTH = (0.02, 0.0)
CAST_W_STRENGTH = 0.30
CAST_DENSITY_X_STRENGTH = (0.25, 0.0)

WELD_X_STRENGTH = 0.02
WELD_ANI_STRENGTH = 18.0

HETEROLAMELLAR_X_STRENGTH = (0.04, 0.26)
HETEROLAMELLAR_ANI_STRENGTH = 7.0
HETEROLAMELLAR_W_STRENGTH = 0.36


lpbf_objective = LayeredBuildLoss(
    stats,
    n_layers=6,
    build_axis_degrees=90.0,
    tilt_degrees=35.0,
    axis_ratio=5.5,
)
cast_objective = CastSlabGuidance(stats)
weld_objective = WeldFusionLoss(stats, axis_ratio=5.5)
heterolamellar_objective = HeteroLamellarGuidance(stats)


MOTIF_SPECS = [
    {
        "name": "Unguided",
        "subtitle": "learned baseline",
        "terms": [],
    },
    {
        "name": "3D-printed steel",
        "subtitle": "alternating build-layer tilt",
        "terms": [
            guidance_term(
                "layered_orientation",
                lpbf_objective,
                x=LPBF_X_STRENGTH,
                ani=LPBF_ANI_STRENGTH,
            )
        ],
    },
    {
        "name": "Cast metal slab",
        "subtitle": "fine / columnar / coarse gradient",
        "terms": [
            guidance_term(
                "cast_shape",
                cast_objective.shape_loss,
                x=CAST_SHAPE_X_STRENGTH,
                ani=CAST_ANI_STRENGTH,
            ),
            guidance_term(
                "cast_relative_weight",
                cast_objective.weight_loss,
                x=CAST_WEIGHT_X_STRENGTH,
                w=CAST_W_STRENGTH,
            ),
            guidance_term(
                "cast_density",
                cast_objective.density_loss,
                x=CAST_DENSITY_X_STRENGTH,
            ),
        ],
    },
    {
        "name": "Copper weld",
        "subtitle": "curved inward growth field",
        "terms": [
            guidance_term(
                "weld_orientation",
                weld_objective,
                x=WELD_X_STRENGTH,
                ani=WELD_ANI_STRENGTH,
            )
        ],
    },
    {
        "name": "Heterogeneous lamella titanium",
        "subtitle": "coarse lamellae in a fine matrix",
        "terms": [
            guidance_term(
                "heterolamellar_topology",
                heterolamellar_objective.band_topology_loss,
                x=HETEROLAMELLAR_X_STRENGTH,
            ),
            guidance_term(
                "heterolamellar_shape",
                heterolamellar_objective.shape_loss,
                ani=HETEROLAMELLAR_ANI_STRENGTH,
            ),
            guidance_term(
                "heterolamellar_relative_weight",
                heterolamellar_objective.weight_loss,
                w=HETEROLAMELLAR_W_STRENGTH,
            ),
        ],
    },
]

print(f"matched trajectories to run: {len(SAMPLE_SEEDS) * len(MOTIF_SPECS)}")


## Generate the matched experiment

Each seed defines one initial Gaussian state. That state is cloned for the
unguided baseline and each guided motif, giving five matched rows by four sample
columns (20 trajectories total).


In [ ]:
def evaluate_terms(state, terms):
    with torch.no_grad():
        return {
            term["name"]: float(term["objective"](state, 1.0).cpu())
            for term in terms
        }


RESULTS = {spec["name"]: [] for spec in MOTIF_SPECS}
HISTORIES = {spec["name"]: [] for spec in MOTIF_SPECS}
METRICS = {spec["name"]: [] for spec in MOTIF_SPECS[1:]}

total_runs = len(SAMPLE_SEEDS) * len(MOTIF_SPECS)
completed_runs = 0

for sample_index, seed in enumerate(SAMPLE_SEEDS, start=1):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    initial_state = make_base_state(NUM_NODES, K_KNN, DEVICE)

    baseline_state = None
    for spec in MOTIF_SPECS:
        final_state, _, history = sample_with_guidance_terms(
            model,
            initial_state,
            guidance_terms=spec["terms"],
            n_steps=N_STEPS,
            save_every=SAVE_EVERY,
            k_knn=K_KNN,
            knn_update_every=KNN_UPDATE_EVERY,
            schedule_start=SCHEDULE_START,
        )
        RESULTS[spec["name"]].append(final_state)
        HISTORIES[spec["name"]].append(history)

        if not spec["terms"]:
            baseline_state = final_state
            diagnostic = ""
        else:
            baseline_losses = evaluate_terms(baseline_state, spec["terms"])
            guided_losses = evaluate_terms(final_state, spec["terms"])
            METRICS[spec["name"]].append(
                {
                    "unguided": baseline_losses,
                    "guided": guided_losses,
                }
            )
            ratios = {
                name: guided_losses[name]
                / max(abs(baseline_losses[name]), 1e-12)
                for name in guided_losses
            }
            diagnostic = " (" + ", ".join(
                f"{name}={value:.3f}x"
                for name, value in ratios.items()
            ) + ")"

        completed_runs += 1
        print(
            f"[{completed_runs:02d}/{total_runs}] sample {sample_index}: "
            f"{spec['name']}{diagnostic}"
        )


## Render the comparison

Generated cells are coloured for visual separation and orientation. The
experimental images occupy only the final comparison column and are not touched
until all sampling is complete.


In [ ]:
def make_apd(normalized_state, stats, pixel_size):
    physical = to_physical_state(normalized_state, stats)
    apd = PyAPD.apd_system(
        X=physical["x"],
        As=physical["A"],
        W=physical["w"],
        pixel_params=(pixel_size, pixel_size),
        device="cpu",
    )
    apd.assemble_pixels()
    pixel_grid = apd.assemble_apd()
    cell_counts = torch.bincount(
        pixel_grid,
        minlength=physical["x"].shape[0],
    )
    apd.set_target_masses(
        cell_counts / float(pixel_size * pixel_size)
    )
    return apd


def render_motif_panel(
    normalized_state,
    stats,
    *,
    pixel_size,
    colour_seed,
):
    apd = make_apd(normalized_state, stats, pixel_size)
    labels = compute_apd(apd).to(torch.long).numpy()

    physical_ani = (
        normalized_state["ani"] * stats["ani_scale"]
    ).detach().cpu()
    magnitude = physical_ani.norm(dim=-1).numpy()
    angles = visible_long_axis_degrees(
        normalized_state,
        stats,
    ).detach().cpu().numpy()

    cell_index = np.arange(physical_ani.shape[0])
    orientation_hue = ((angles + 90.0) / 180.0) % 1.0
    fallback_hue = (
        0.61803398875 * cell_index + 0.037 * colour_seed
    ) % 1.0
    hue = np.where(magnitude < 0.12, fallback_hue, orientation_hue)
    elongation = np.clip(magnitude / math.log(7.0), 0.0, 1.0)
    saturation = 0.42 + 0.24 * elongation
    value = 0.92 - 0.08 * (
        ((37 * cell_index + colour_seed) % 17) / 16.0
    )
    colours = mcolors.hsv_to_rgb(
        np.column_stack([hue, saturation, value])
    )
    rgb = colours[labels]

    edge = np.zeros(labels.shape, dtype=bool)
    edge[:-1, :] |= labels[:-1, :] != labels[1:, :]
    edge[:, :-1] |= labels[:, :-1] != labels[:, 1:]
    rgb[edge] = 0.10
    return rgb


In [ ]:
REFERENCE_DIRECTORY = (
    ROOT / "examples_for_guidance" / "reference_microstructures"
)

REFERENCE_SOURCES = {
    "Unguided": {
        "path": REFERENCE_DIRECTORY / "unguided.png",
        "label": None,
        "crop": None,
    },
    "3D-printed steel": {
        "path": REFERENCE_DIRECTORY / "lpbf.png",
        "label": None,
        "crop": None,
    },
    "Cast metal slab": {
        "path": REFERENCE_DIRECTORY / "cast.png",
        "label": None,
        "crop": None,
    },
    "Copper weld": {
        "path": REFERENCE_DIRECTORY / "weld.png",
        "label": None,
        "crop": None,
    },
    "Heterogeneous lamella titanium": {
        "path": REFERENCE_DIRECTORY / "heterogeneous_lamella.png",
        "label": None,
        "crop": None,
    },
}

missing_references = [
    source["path"]
    for source in REFERENCE_SOURCES.values()
    if not source["path"].exists()
]
if missing_references:
    raise FileNotFoundError(
        "Missing experimental reference files: "
        + ", ".join(str(path) for path in missing_references)
    )


In [ ]:
def cropped_reference_image(source):
    image = plt.imread(source["path"])
    crop = source.get("crop")
    if crop is None:
        return image
    x0, x1, y0, y1 = crop
    height, width = image.shape[:2]
    left = int(round(x0 * width))
    right = int(round(x1 * width))
    top = int(round(y0 * height))
    bottom = int(round(y1 * height))
    return image[top:bottom, left:right]


def plot_motif_rows_with_references(
    motif_specs,
    results,
    stats,
    seeds,
    reference_sources,
    *,
    pixel_size=250,
    output_path=None,
):
    n_rows = len(motif_specs)
    n_generated_columns = len(seeds)
    n_columns = n_generated_columns + 1
    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(3.05 * n_columns, 3.05 * n_rows),
        squeeze=False,
    )

    for row, spec in enumerate(motif_specs):
        name = spec["name"]
        states = results[name]
        if len(states) != n_generated_columns:
            raise ValueError(
                f"{name} has {len(states)} states; "
                f"expected {n_generated_columns}"
            )
        if name not in reference_sources:
            raise KeyError(f"No experimental reference for {name}")

        for column, (seed, state) in enumerate(zip(seeds, states)):
            axis = axes[row, column]
            rgb = render_motif_panel(
                state,
                stats,
                pixel_size=pixel_size,
                colour_seed=seed,
            )
            axis.imshow(rgb, origin="lower", interpolation="nearest")
            axis.set_xticks([])
            axis.set_yticks([])

        source = reference_sources[name]
        reference_axis = axes[row, -1]
        reference_axis.imshow(
            cropped_reference_image(source),
            interpolation="nearest",
        )
        reference_axis.set_xticks([])
        reference_axis.set_yticks([])
        reference_axis.set_xlabel(
            source["label"],
            fontsize=8,
            color="0.25",
            labelpad=5,
        )
        for spine in reference_axis.spines.values():
            spine.set_color("0.35")
            spine.set_linewidth(1.2)

    # Reserve fixed gutters so labels align independently of image aspect ratio.
    fig.tight_layout(rect=(0.15, 0.0, 1.0, 0.965))

    for row, spec in enumerate(motif_specs):
        position = axes[row, 0].get_position()
        row_label = spec["name"]
        if len(row_label) > 22:
            words = row_label.split()
            split_at = min(
                range(1, len(words)),
                key=lambda index: abs(
                    len(" ".join(words[:index]))
                    - len(" ".join(words[index:]))
                ),
            )
            row_label = (
                " ".join(words[:split_at])
                + "\n"
                + " ".join(words[split_at:])
            )
        fig.text(
            0.075,
            0.5 * (position.y0 + position.y1),
            row_label,
            fontsize=14,
            fontweight="semibold",
            color="0.08",
            ha="center",
            va="center",
            linespacing=1.15,
        )

    column_titles = [
        *(f"Generated sample {column + 1}" for column in range(n_generated_columns)),
        "Experimental reference",
    ]
    for column, title in enumerate(column_titles):
        position = axes[0, column].get_position()
        fig.text(
            0.5 * (position.x0 + position.x1),
            0.974,
            title,
            fontsize=13,
            fontweight="bold",
            color="0.08",
            ha="center",
            va="bottom",
        )
    if output_path is not None:
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
        print(f"saved {output_path}")
    plt.show()
    return fig


In [ ]:
comparison_figure = plot_motif_rows_with_references(
    MOTIF_SPECS,
    RESULTS,
    stats,
    SAMPLE_SEEDS,
    REFERENCE_SOURCES,
    pixel_size=PLOT_PIXEL_SIZE,
    output_path=MOTIF_FIGURE_PATH if SAVE_MOTIF_FIGURE else None,
)


## Reference-image provenance

From top to bottom, the comparison uses the following sources:

1. Unguided APD reference: M. Buze, J. Feydy, S. M. Roper, K. Sedighiani, and D. P. Bourne. Anisotropic power diagrams for polycrystal modelling: Efficient generation of curved grains via optimal transport. URL https://www.sciencedirect.com/science/article/pii/S092702562400538X.
2. 3D-printed steel: Yanis Balit, Eric Charkaluk, and Andrei Constantinescu. Digital image correlation for microstructural analysis of deformation pattern in additively manufactured 316L thin walls. URL https://linkinghub.elsevier.com/retrieve/pii/S2214860419305469.
3. Cast slab schematic: Robert E. Reed-Hill. Physical Metallurgy Principles. D. Van Nostrand Company, New York, 2nd edition, 1973.
4. Copper weld: Kati Savolainen, Tapio Saukkonen, and Hannu Hänninen. Localization of plastic deformation in copper canisters for spent nuclear fuel. URL https://www.scirp.net/journal/paperinformation?paperid=16567.
5. Heterogeneous lamella titanium: Xiaolei Wu, Muxin Yang, Fuping Yuan, Guilin Wu, Yujie Wei, Xiaoxu Huang, and Yuntian Zhu. Heterogeneous lamella structure unites ultrafine-grain strength with coarse-grain ductility. URL https://www.pnas.org/doi/abs/10.1073/pnas.1517193112.
